In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
import os
import numpy as np

base_path = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET'

# 1. Advanced Native Transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15), 
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # Shifts face around
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.5, scale=(0.02, 0.1)) # Regularization
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 2. Balanced Loading
image_datasets = {x: datasets.ImageFolder(os.path.join(base_path, x), data_transforms[x]) for x in ['train', 'test']}
targets = image_datasets['train'].targets
class_weights = 1.0 / np.bincount(targets)
sampler = WeightedRandomSampler([class_weights[t] for t in targets], len(targets))

# Batch size lowered to 32 to accommodate the larger B2 model
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=32, sampler=sampler, num_workers=4),
    'test': DataLoader(image_datasets['test'], batch_size=32, shuffle=False, num_workers=4)
}

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"✅ Data Pipeline Ready on {device} (Pure PyTorch)")

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(label_smoothing=0.1)(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        return focal_loss

def build_emotion_model():
    # Upgrade to B2 for more detail
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
    for param in model.features.parameters():
        param.requires_grad = False
    
    # B2 has 1408 output features from the backbone
    num_ftrs = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(512, 7)
    )
    return model.to(device)

model = build_emotion_model()
criterion = FocalLoss(gamma=2) # Using Focal Loss instead of Cross Entropy
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

print("✅ Model initialized with EfficientNet-B2 and Focal Loss.")

In [ ]:
import time
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

def train_model_advanced(model, criterion, optimizer, scheduler, num_epochs, use_swa=False):
    since = time.time()
    best_acc = 0.0
    
    # Initialize SWA if needed
    if use_swa:
        swa_model = AveragedModel(model)
        # SWA usually starts 75% of the way through the training
        swa_start = int(num_epochs * 0.9)
        swa_scheduler = SWALR(optimizer, swa_lr=1e-5)

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        for phase in ['train', 'test']:
            model.train() if phase == 'train' else model.eval()
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_corrects += torch.sum(preds == labels.data)

            epoch_acc = running_corrects.double() / len(image_datasets[phase])
            print(f'{phase.capitalize()} Acc: {epoch_acc:.4f}')

            if phase == 'test':
                if not use_swa: scheduler.step(epoch_acc)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    torch.save(model.state_dict(), 'best_temp_weights.pth')
        
        # Apply SWA updates in the final epochs
        if use_swa and epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_scheduler.step()

    if use_swa:
        update_bn(dataloaders['train'], swa_model, device=device)
        return swa_model
    
    model.load_state_dict(torch.load('best_temp_weights.pth'))
    return model

In [ ]:
# --- STAGE 1: Head Training ---
print("🚀 Phase 1: Classifier Training...")
trained_model = train_model_advanced(model, criterion, optimizer, scheduler, num_epochs=10)

# --- STAGE 2: Fine-Tuning + SWA ---
print("🧠 Phase 2: Global Fine-Tuning with SWA...")
for param in trained_model.parameters():
    param.requires_grad = True

# Keeping your proven learning rate flat for the entire model
optimizer_ft = optim.Adam(trained_model.parameters(), lr=1e-5)

# SWA is enabled for this final, high-precision pass
final_model = train_model_advanced(trained_model, criterion, optimizer_ft, None, num_epochs=35, use_swa=True)

# Save the final masterpiece
torch.save(final_model.state_dict(), '/kaggle/working/emotion_model_ULTIMATE_V2.pth')
print("✅ ALL SYSTEMS GO! Download 'emotion_model_ULTIMATE_V2.pth' for your demo.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import torch

# --- 1. SETUP ---
# Grab the dynamic class names directly from your dataset folders
class_names = image_datasets['train'].classes 

def evaluate_ultimate_model(model, dataloader):
    print("🔍 Running Deep Analytics on the Test Set. Please wait...")
    model.eval() # Lock the model
    
    all_preds = []
    all_labels = []

    # --- 2. GATHER PREDICTIONS ---
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # Move data off the GPU to standard RAM for sklearn
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # --- 3. CLASSIFICATION REPORT ---
    print("\n" + "="*50)
    print("📊 FINAL CLASSIFICATION REPORT")
    print("="*50)
    # This generates Precision, Recall, and F1 for every single emotion
    report = classification_report(all_labels, all_preds, target_names=class_names)
    print(report)

    # --- 4. CONFUSION MATRIX VISUALIZATION ---
    cm = confusion_matrix(all_labels, all_preds)
    
    # Calculate percentages for the matrix
    cm_percentages = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(12, 10))
    # We use 'Blues' for the color map. Darker blue = higher accuracy
    sns.heatmap(cm_percentages, annot=True, fmt='.2f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 12})
    
    plt.ylabel('TRUE Emotion (What the image actually is)', fontsize=14, fontweight='bold')
    plt.xlabel('PREDICTED Emotion (What your model guessed)', fontsize=14, fontweight='bold')
    plt.title('Emotion Recognition Confusion Matrix (Accuracy %)', fontsize=16, fontweight='bold', pad=20)
    
    # Rotate the bottom labels so they don't overlap
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    plt.show()

# --- 5. EXECUTE ---
# Run this using your newly saved masterpiece
evaluate_ultimate_model(final_model, dataloaders['test'])